# RAG from Scratch to Production — Episode 10/20
## Reranking: does the second opinion actually help?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/10_reranking.ipynb)

Every search so far worked the same way: turn the question into a vector (or words), turn each chunk into a vector, compare. The question and the chunk **never meet**.

A **reranker** (a *cross-encoder*) reads the question and a chunk **together**, like a person would, and scores how well that chunk answers that question. It's slower, so it only re-sorts a shortlist. The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) reranks the hybrid search's candidates and keeps the top 3.

The standard advice is "always add a reranker". **Let's measure it.**

**In this episode:**
1. See the difference between embeddings and a cross-encoder.
2. Test production's reranker on two handbooks, including one built to be hard.
3. Try two bigger rerankers.
4. Measure the cost.
5. Find what the reranker is **actually** good at.

**No API key needed.**

## 0. Setup

In [ ]:
%pip install -q sentence-transformers rank-bm25

In [1]:
import re, math, time
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

embedder = SentenceTransformer("all-MiniLM-L6-v2")                      # production's embedding model
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")         # production's reranker

def title(chunk: str) -> str:
    return chunk.split("\n")[0]

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

def minmax(x):
    return (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else np.zeros_like(x)

def sigmoid(x: float) -> float:
    return 1 / (1 + math.exp(-x))

## 1. Two ways to compare a question with a chunk

| | Embeddings (bi-encoder) | Reranker (cross-encoder) |
|---|---|---|
| Reads | Question and chunk **separately** | Question and chunk **together** |
| Chunk work | Done once, at upload | Done again for **every question** |
| Speed | Fast: one multiplication per chunk | Slow: a full model run per chunk |
| Used for | Searching everything | Re-sorting a shortlist |

Here's the reranker scoring one question against two similar chunks:

In [2]:
question = "How many leave days do part-time employees get?"
pair_chunks = [
    "Full-time employees receive 24 days of paid annual leave per year.",
    "Part-time employees receive 12 days of paid annual leave per year.",
]
raw = reranker.predict([[question, c] for c in pair_chunks])
for chunk, score in zip(pair_chunks, raw):
    print(f"raw {score:6.2f}   as 0–1 {sigmoid(score):.3f}   {chunk}")

raw   6.84   as 0–1 0.999   Full-time employees receive 24 days of paid annual leave per year.
raw   9.40   as 0–1 1.000   Part-time employees receive 12 days of paid annual leave per year.


The model outputs a raw score (any number). Production turns it into a 0–1 score with a **sigmoid**, as above.

Notice: both chunks score about **1.0** after the sigmoid, but the raw scores clearly prefer the **part-time** chunk (9.40 vs 6.84). The sigmoid squashes differences between strong matches. Fine for sorting, but keep it in mind when reading 0–1 scores.

## 2. The pipeline: search, then rerank

Episode 8's best hybrid search picks a **shortlist of 5**, the reranker re-sorts it, and the top one wins. This is production's shape (`app/services/reranker.py`), minus the threshold, which gets its own episode next.

In [3]:
def build_index(chunks):
    vectors = embedder.encode(chunks, normalize_embeddings=True)
    bm25 = BM25Okapi([tokenize(c) for c in chunks])
    return vectors, bm25


def embedding_scores(question, index):
    return index[0] @ embedder.encode(question, normalize_embeddings=True)


def hybrid_scores(question, index):
    keyword = np.array(index[1].get_scores(tokenize(question)))
    return 0.8 * minmax(embedding_scores(question, index)) + 0.2 * minmax(keyword)


def rerank(question, chunks, shortlist, model=reranker):
    scores = model.predict([[question, chunks[i]] for i in shortlist])
    order = np.argsort(-scores)
    return [shortlist[i] for i in order], [sigmoid(float(scores[i])) for i in order]


def run_test(name, chunks, questions, models=None, verbose=True):
    """Top-1 accuracy of embeddings, hybrid, and hybrid + reranker(s)."""
    models = models or {"+ reranker": reranker}
    index = build_index(chunks)
    totals = {"embeddings": 0, "hybrid": 0, **{m: 0 for m in models}}
    if verbose:
        print(f"── {name}")
    for question, answer in questions:
        picks = {"embeddings": int(np.argmax(embedding_scores(question, index)))}
        hybrid = hybrid_scores(question, index)
        picks["hybrid"] = int(np.argmax(hybrid))
        shortlist = list(np.argsort(-hybrid)[:5])
        for m, model in models.items():
            picks[m] = rerank(question, chunks, shortlist, model)[0][0]
        row = ""
        for k, i in picks.items():
            hit = title(CHUNK_TITLES[name][i]) == answer
            totals[k] += hit
            row += f"{'✅' if hit else '❌'} "
        if verbose:
            print(f"   {row}  {question}")
    print(f"   TOTAL: " + "   ".join(f"{k} {v}/{len(questions)}" for k, v in totals.items()))
    return totals

## 3. Test 1: the handbook from earlier episodes

8 sections, each about a different topic, and 14 questions.

In [4]:
HANDBOOK = [
    "Remote Work\nEmployees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams. Remote days must be logged in the HR portal by Friday of the previous week.",
    "Annual Leave\nFull-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over into the next year. Book leave with form HR-12.",
    "Parental Leave\nNew parents receive 16 weeks of fully paid leave. Submit form HR-17 at least 8 weeks before the expected start date.",
    "Expenses\nThe meal allowance during business travel is capped at 40 USD per day. Travel expenses are claimed with form EXP-22 within 30 days. Claims above 500 USD need the finance director.",
    "Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses and conferences. It does not roll over. Requests are approved by your direct manager.",
    "Equipment\nEvery employee receives a ThinkPad X1 Carbon laptop, refreshed every 3 years. Lost or stolen devices must be reported to IT within 24 hours.",
    "IT Support\nIf the VPN shows error E-4031, your access token has expired: reset it in the Okta portal. For error E-4032, contact the IT helpdesk.",
    "Security\nMulti-factor authentication is mandatory on all company accounts. Passwords must be at least 14 characters long and must never be reused.",
]
HANDBOOK_QUESTIONS = [
    ("How many weeks of leave do new parents get?",          "Parental Leave"),
    ("How many vacation days carry over to next year?",      "Annual Leave"),
    ("Does the learning budget carry over to next year?",    "Learning Budget"),
    ("Who has to approve expense claims over 500 USD?",      "Expenses"),
    ("Who approves my course and conference requests?",      "Learning Budget"),
    ("What happens if my VPN access token expired?",         "IT Support"),
    ("How fast do I report a stolen phone?",                 "Equipment"),
    ("Which day must I be in the office?",                   "Remote Work"),
    ("Can I reuse an old password?",                         "Security"),
    ("What's the deadline for logging remote days?",         "Remote Work"),
    ("How long do I have to submit a travel expense claim?", "Expenses"),
    ("When must the parental leave form be submitted?",      "Parental Leave"),
    ("Can I save unused training money for next year?",      "Learning Budget"),
    ("What do I get for food when travelling for work?",     "Expenses"),
]
CHUNK_TITLES = {"Test 1: handbook": HANDBOOK}

print("   emb hyb rr")
test1 = run_test("Test 1: handbook", HANDBOOK, HANDBOOK_QUESTIONS)

   emb hyb rr
── Test 1: handbook
   ✅ ✅ ✅   How many weeks of leave do new parents get?
   ✅ ✅ ✅   How many vacation days carry over to next year?
   ✅ ✅ ✅   Does the learning budget carry over to next year?
   ✅ ✅ ✅   Who has to approve expense claims over 500 USD?
   ✅ ✅ ✅   Who approves my course and conference requests?
   ✅ ✅ ✅   What happens if my VPN access token expired?
   ✅ ✅ ✅   How fast do I report a stolen phone?
   ✅ ✅ ✅   Which day must I be in the office?
   ✅ ✅ ✅   Can I reuse an old password?
   ✅ ✅ ✅   What's the deadline for logging remote days?
   ✅ ✅ ✅   How long do I have to submit a travel expense claim?


   ✅ ✅ ✅   When must the parental leave form be submitted?


   ✅ ✅ ❌   Can I save unused training money for next year?
   ✅ ✅ ✅   What do I get for food when travelling for work?
   TOTAL: embeddings 14/14   hybrid 14/14   + reranker 13/14


Embeddings and hybrid search already put the right section first every time. The reranker can't improve on perfect, and it **lost one**: "training money".

That's not a fair test, though. Each section here is about a different topic, so even a rough search can't confuse them. Rerankers are supposed to shine when chunks are **similar** and the details decide.

## 4. Test 2: a handbook built to be hard

Pairs of sections that are nearly identical except for one detail: full-time vs part-time, birth vs adoptive parents, domestic vs international trips, engineering vs sales laptops. Real policy documents are full of this.

In [5]:
HARD_HANDBOOK = [
    "Annual Leave: Full-time Employees\nFull-time employees receive 24 days of paid annual leave per year. Up to 5 unused days may be carried over into the next year.",
    "Annual Leave: Part-time Employees\nPart-time employees receive 12 days of paid annual leave per year. Unused days cannot be carried over.",
    "Parental Leave: Birth Parents\nBirth parents receive 16 weeks of fully paid leave. Submit form HR-17 at least 8 weeks before the expected due date.",
    "Parental Leave: Adoptive Parents\nAdoptive parents receive 10 weeks of fully paid leave, starting from the day the child joins the family. Submit form HR-18.",
    "Travel Expenses: Domestic Trips\nOn trips inside the country, the meal allowance is 40 USD per day. Hotels are booked through the travel portal.",
    "Travel Expenses: International Trips\nOn trips abroad, the meal allowance is 75 USD per day. International flights over 6 hours may be booked in premium economy.",
    "Remote Work: Office Staff\nOffice staff may work remotely up to 2 days per week. Wednesday is a mandatory in-office day.",
    "Remote Work: Field Engineers\nField engineers work fully remotely and visit the office once per quarter for planning.",
    "Laptops: Engineering\nEngineers receive a ThinkPad X1 Carbon, replaced every 3 years.",
    "Laptops: Sales\nSales staff receive a MacBook Air, replaced every 4 years.",
    "Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses and conferences. It does not roll over.",
    "Security\nMulti-factor authentication is mandatory. Passwords must be at least 14 characters long and must never be reused.",
]
HARD_QUESTIONS = [
    ("How many leave days do part-time employees get?",           "Annual Leave: Part-time Employees"),
    ("Can part-timers carry unused leave into next year?",        "Annual Leave: Part-time Employees"),
    ("How much paid leave do I get if I adopt a child?",          "Parental Leave: Adoptive Parents"),
    ("Which form do adoptive parents submit?",                    "Parental Leave: Adoptive Parents"),
    ("What's the meal allowance on a trip to Germany?",           "Travel Expenses: International Trips"),
    ("Can I fly premium economy to Japan?",                       "Travel Expenses: International Trips"),
    ("How often do field engineers come to the office?",          "Remote Work: Field Engineers"),
    ("What laptop does the sales team get?",                      "Laptops: Sales"),
    ("How often are sales laptops replaced?",                     "Laptops: Sales"),
    ("What is the meal allowance for a trip within the country?", "Travel Expenses: Domestic Trips"),
    ("How many weeks of leave does a birth parent get?",          "Parental Leave: Birth Parents"),
    ("How many days can office staff work from home?",            "Remote Work: Office Staff"),
]
CHUNK_TITLES["Test 2: hard handbook"] = HARD_HANDBOOK

print("   emb hyb rr")
test2 = run_test("Test 2: hard handbook", HARD_HANDBOOK, HARD_QUESTIONS)

   emb hyb rr
── Test 2: hard handbook
   ✅ ✅ ✅   How many leave days do part-time employees get?
   ✅ ✅ ❌   Can part-timers carry unused leave into next year?
   ✅ ✅ ✅   How much paid leave do I get if I adopt a child?
   ✅ ✅ ✅   Which form do adoptive parents submit?
   ✅ ❌ ❌   What's the meal allowance on a trip to Germany?
   ✅ ✅ ✅   Can I fly premium economy to Japan?
   ✅ ✅ ✅   How often do field engineers come to the office?
   ✅ ✅ ✅   What laptop does the sales team get?
   ✅ ✅ ✅   How often are sales laptops replaced?
   ✅ ✅ ✅   What is the meal allowance for a trip within the country?
   ✅ ✅ ✅   How many weeks of leave does a birth parent get?


   ✅ ✅ ✅   How many days can office staff work from home?
   TOTAL: embeddings 12/12   hybrid 11/12   + reranker 10/12


**The reranker made it worse.** Look at its two misses:
- *"Can **part-timers** carry unused leave?"* → it picked **full-time**, because that section is the one that talks about carrying days over.
- *"Meal allowance on a trip to **Germany**?"* → it picked **domestic**. It doesn't connect "Germany" with "abroad".

Production's reranker is the smallest common one (about 90 MB), trained on web-search questions. Is that the problem?

## 5. Bigger rerankers, and chunks without titles

Two stronger free rerankers, and a second version of the hard handbook **without section titles**: real 1,000-character PDF chunks often don't start with a helpful heading. This downloads about 1.2 GB.

In [6]:
MODELS = {
    "MiniLM-L-6 (production)": reranker,
    "MiniLM-L-12":             CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2"),
    "bge-reranker-base":       CrossEncoder("BAAI/bge-reranker-base"),
}

no_titles = [c.split("\n", 1)[1] for c in HARD_HANDBOOK]
CHUNK_TITLES["with titles"] = HARD_HANDBOOK
CHUNK_TITLES["without titles"] = HARD_HANDBOOK        # same order, so answers map to the same titles

for name, chunks in [("with titles", HARD_HANDBOOK), ("without titles", no_titles)]:
    run_test(name, chunks, HARD_QUESTIONS, models=MODELS, verbose=False)

   TOTAL: embeddings 12/12   hybrid 11/12   MiniLM-L-6 (production) 10/12   MiniLM-L-12 11/12   bge-reranker-base 12/12


   TOTAL: embeddings 11/12   hybrid 11/12   MiniLM-L-6 (production) 10/12   MiniLM-L-12 11/12   bge-reranker-base 9/12


On this test, **no reranker beats plain embeddings**:
- The biggest one (bge-reranker-base) ties with embeddings when chunks have titles, and does **worse** without them.
- The medium one is the most consistent, but never better than embeddings.

A fair caveat: these are small test sets (8 and 12 chunks). Here, the first-stage search usually already has the right chunk **first**. A reranker earns its place when there are thousands of similar chunks and the right one is in the top 10 but not at the top. That's a situation you should **measure on your own documents**, not assume.

## 6. The cost

In [7]:
q = "How fast do I report a stolen phone?"
for n in [10, 100, 1000]:
    pairs = [[q, HANDBOOK[i % len(HANDBOOK)]] for i in range(n)]
    start = time.perf_counter()
    reranker.predict(pairs)
    print(f"reranking {n:>5} chunks: {(time.perf_counter() - start) * 1000:6.0f} ms")

start = time.perf_counter()
for _ in range(20):
    embedder.encode(q)
print(f"embedding the question once: {(time.perf_counter() - start) / 20 * 1000:.1f} ms  (then searching every chunk is one multiplication)")

reranking    10 chunks:     12 ms
reranking   100 chunks:     74 ms


reranking  1000 chunks:    518 ms
embedding the question once: 3.3 ms  (then searching every chunk is one multiplication)


The reranker runs the model **once per chunk, per question**. That's why it only ever sees a shortlist: on 10 chunks it's cheap, on a whole collection it's far too slow.

## 7. What a reranker is actually good at

Episode 8 asked the handbook questions it can't answer ("What is the capital of France?"), and **no retrieval score** could tell them apart from real questions. Let's ask the reranker:

In [8]:
index = build_index(HANDBOOK)

def best_rerank_score(question):
    shortlist = list(np.argsort(-hybrid_scores(question, index))[:5])
    return rerank(question, HANDBOOK, shortlist)[1][0]

print("Can't be answered:")
for q in ["What is the capital of France?", "Who won the football World Cup in 2022?",
          "What is the dress code for the office party?", "How do I bake sourdough bread?"]:
    print(f"   {best_rerank_score(q):.4f}   {q}")

print("\nCan be answered:")
for q, _ in HANDBOOK_QUESTIONS:
    print(f"   {best_rerank_score(q):.4f}   {q}")

Can't be answered:
   0.0000   What is the capital of France?
   0.0000   Who won the football World Cup in 2022?
   0.0000   What is the dress code for the office party?
   0.0000   How do I bake sourdough bread?
Can be answered:
   0.9999   How many weeks of leave do new parents get?
   0.9874   How many vacation days carry over to next year?
   0.9696   Does the learning budget carry over to next year?
   0.9684   Who has to approve expense claims over 500 USD?
   0.9950   Who approves my course and conference requests?
   0.9995   What happens if my VPN access token expired?
   0.7197   How fast do I report a stolen phone?
   0.9873   Which day must I be in the office?
   0.0192   Can I reuse an old password?


   0.9321   What's the deadline for logging remote days?
   0.9870   How long do I have to submit a travel expense claim?
   0.9995   When must the parental leave form be submitted?
   0.0063   Can I save unused training money for next year?
   0.0324   What do I get for food when travelling for work?


**This is the reranker's real superpower.** Off-topic questions score about **0.0000**. Most real questions score close to **1**.

Embedding and hybrid scores only tell you which chunk is **more** relevant than the others. The reranker's score says whether a chunk is relevant **at all**. That's what lets a RAG system say *"I don't have that information"* instead of answering from the closest chunk it found.

But look closely at the real questions: **3 of the 14 correct matches** also score below 0.04, like *"Can I reuse an old password?"* So where do you draw the line between "relevant" and "not"? Production uses **0.20**. Draw it too high, and correct answers get thrown away. Draw it too low, and junk gets through.

That's **Episode 11: the reranker threshold bug**.

## 8. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) (`app/services/reranker.py`):
- Reranks the hybrid search's candidates with `cross-encoder/ms-marco-MiniLM-L-6-v2`, against the **rewritten** question (Episode 9).
- Turns raw scores into 0–1 with a sigmoid, keeps chunks scoring at least **0.20**, and passes the **top 3** to the LLM.

## Recap

| | Embeddings | Reranker (cross-encoder) |
|---|---|---|
| Reads | Question and chunk separately | Together |
| Ranking on our tests | Already right almost every time | No better, sometimes worse |
| Cost | Tiny per chunk | A model run per chunk, per question |
| Can it say "nothing here is relevant"? | ❌ | ✅ off-topic ≈ 0.0000 |

**The rule:** don't add a reranker because everyone does. Measure whether it improves **your** ranking, and use its scores for what they're good at: deciding when **nothing** is relevant.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)